# 步骤 04 · 时频图 —— 第一次看见那个节奏

**这一节的产出：图 4a（时频图，周期性条纹）+ 图 4b（带内能量随时间起伏，及其频谱）。**

## 上一节把问题定位了

步骤 03 查明了两件事：

1. 故障的能量**不在** 105.9 Hz，在 **2000–4000 Hz**（外圈占 97.4%）
2. 低频段那些峰是机器谱线，健康记录里也有，**靠它们会误判**

并且给出了解释：**105.9 Hz 是冲击"多久来一次"，不是冲击"本身的频率"。**

## 那么这一节要验证的假设是

> 如果"每秒 105.9 次撞击"这件事真的在发生，那么 2000–4000 Hz 那片能量**不应该是恒定的**。它应该每秒钟忽强忽弱 105.9 次 —— 撞一下亮一下，衰减，再撞一下再亮。

**这一节就是去看它到底闪不闪。**

FFT 做不到这件事，因为它把整整 10 秒揉成一张频谱，**时间信息全丢了**。它能告诉你"有 3000 Hz 的成分"，但不能告诉你"这个成分在第 0.05 秒时强、第 0.06 秒时弱"。

所以需要一个既看频率、又保留时间的工具：**短时傅里叶变换**。

---
## 1. 短时傅里叶变换（STFT）是什么

想法特别朴素：

> **别对整段 10 秒做 FFT。切成很多小段，每段单独做一次 FFT，然后把结果排成一张图。**

- 横轴：时间（第几小段）
- 纵轴：频率
- 颜色：那个时刻、那个频率上有多强

这张图叫**时频图**或**语谱图**（spectrogram）。

> 你其实见过它 —— 音乐播放器里跳动的频谱柱、语音识别的可视化、Shazam 听歌识曲用的都是它。

### 核心权衡：时间精度和频率精度不可兼得

这是 STFT 唯一需要真正理解的事。

**窗口开得短** → 能分辨"这一瞬间"发生了什么，但一小段里装不下几个周期，**频率算不准**
**窗口开得长** → 频率算得准，但"这一瞬间"被抹成了一大片，**时间分辨率差**

$$\Delta t \cdot \Delta f \approx 1$$

窗长 64 点 = 5.3 ms，频率分辨率就是 1/0.0053 ≈ **187.5 Hz**。想要 10 Hz 的频率分辨率？窗口得开到 0.1 秒，那就看不出 9.4 ms 一次的冲击了。

> 这个限制和信号处理算法好坏无关，它是傅里叶变换的数学性质。物理里的不确定性原理是同一个数学事实的另一个身份。

**对我们来说，时间精度远比频率精度重要** —— 我们只需要知道"能量在 2–4 kHz 这一大片里"，不需要知道是 2871 还是 2873 Hz。但我们非常需要看清 9.4 ms 一次的节奏。

**所以窗口要开短。**

---
## 2. 准备

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import cwru_io

DATA = ROOT / "data"
FIGURES = ROOT / "figures"

normal = cwru_io.load_baseline(DATA / "normal_1hp_98.mat",    name="Healthy")
outer  = cwru_io.load(DATA / "OR007at6_1hp_131.mat", name="Outer race")
inner  = cwru_io.load(DATA / "IR007_1hp_106.mat",    name="Inner race")
ball   = cwru_io.load(DATA / "B007_1hp_119.mat",     name="Ball")

signals = [normal, outer, inner, ball]
N = min(len(s.x) for s in signals)
FS = cwru_io.FS

# 每条记录"该查哪个频率"
TARGET = {"Healthy": "BPFO", "Outer race": "BPFO",
          "Inner race": "BPFI", "Ball": "BSF"}

for s in signals:
    f0 = s.fault_freqs()[TARGET[s.name]]
    print(f"{s.name:<12} 查 {TARGET[s.name]} = {f0:6.1f} Hz"
          f"   冲击间隔 {1000/f0:5.2f} ms")

---
## 3. 参数怎么选 —— 一个会让整步失效的坑

STFT 有两个参数：**窗长**（`nperseg`）和**重叠**（`noverlap`）。它们之间的差 `hop = nperseg - noverlap` 决定了**每隔多久算一帧**。

关键在于：**这些帧构成了一个新的信号 —— "能量随时间的变化"。而这个新信号也有自己的采样率。**

$$f_{s,\text{包络}} = \frac{f_s}{\text{hop}}$$

上一节讲过奈奎斯特：**要看清一个 105.9 Hz 的节奏，采样率至少得是 211.8 Hz，实践中要留 4 倍余量。**

下面这格把常见参数组合算一遍，看哪些根本不够用。

In [ ]:
BPFO = outer.fault_freqs()["BPFO"]

print(f"要看清的节奏: {BPFO:.1f} Hz，奈奎斯特要求包络采样率 > {2*BPFO:.1f} Hz\n")
print(f"{'窗长':>6}{'重叠':>8}{'hop':>6}{'窗覆盖冲击周期':>16}{'包络采样率':>13}   判定")
print("-" * 72)
for nperseg, frac in [(256, 0.50), (256, 0.75), (256, 0.875),
                      (128, 0.875), (64, 0.75), (64, 0.875)]:
    hop = int(nperseg * (1 - frac))
    env_fs = FS / hop
    cycles = nperseg / (FS / BPFO)
    if env_fs < 2 * BPFO:
        verdict = "X 混叠，节奏被抹掉"
    elif env_fs < 4 * BPFO:
        verdict = "~ 勉强"
    else:
        verdict = "OK"
    print(f"{nperseg:>6}{frac:>8.1%}{hop:>6}{cycles:>16.2f}{env_fs:>13.1f}   {verdict}")

### 读这张表

**第一行就是常见教程里给的参数：窗长 256、重叠 50%。**

hop = 128，包络采样率只有 **93.8 Hz** —— 比要看的 105.9 Hz 还低。它的奈奎斯特上限是 46.9 Hz。

**105.9 Hz 在这套参数下根本不存在于频率轴上。** 不是"看不清"，是"被彻底抹掉"。我第一次按这个参数算，程序直接因为"频率轴上找不到 105.9 Hz"而报错。

还有第四列：窗长 256 覆盖 **2.26 个冲击周期** —— 一个窗里装了两次多撞击，它们被平均成一团，闪烁本身就被窗口抹平了。

**我们选窗长 64、重叠 87.5%（hop = 8）**：窗只覆盖 0.56 个冲击周期（一个窗里最多一次撞击），包络采样率 1500 Hz，是 105.9 Hz 的 14 倍。绰绰有余。

> **这一节最实用的一条经验**：STFT 的参数不是"照抄一组就行"，它必须**由你要看的那个节奏反推出来**。先算出目标频率，再定 hop，最后定窗长。

---
## 4. 图 4a：时频图

现在画。只显示 **0.1 秒**（0.10 到 0.20 秒之间那段），因为要看清单次冲击。

颜色用**分贝**（dB，对数刻度）。理由和上一节频谱用对数轴一样：能量跨了好几个数量级，线性刻度下弱的全黑。四张图用**同一个色标范围**，否则没法横向比 —— 和图 1 的 `sharey` 是同一个道理。

In [ ]:
NPERSEG, HOP = 64, 8
T0, T1 = 0.10, 0.20            # 显示哪一段时间

def spectrogram(x, nperseg=NPERSEG, hop=HOP):
    f, t, Z = stft(x, FS, nperseg=nperseg, noverlap=nperseg - hop,
                   window="hann", boundary=None, padded=False)
    return f, t, np.abs(Z) ** 2          # 功率

grams = {s.name: spectrogram(s.x[:N]) for s in signals}

# 统一色标：以最强的那张为上限，往下 55 dB
vmax = max(10 * np.log10(P.max()) for _, _, P in grams.values())
vmin = vmax - 55

fig4a, axes = plt.subplots(4, 1, figsize=(11, 9.5), sharex=True, sharey=True)

for ax, s in zip(axes, signals):
    f, t, P = grams[s.name]
    m = (t >= T0) & (t <= T1)
    im = ax.pcolormesh(t[m], f, 10 * np.log10(P[:, m] + 1e-20),
                       vmin=vmin, vmax=vmax, cmap="magma", shading="auto")
    ax.axhline(2000, color="w", linewidth=0.6, alpha=0.5, linestyle=":")
    ax.axhline(4000, color="w", linewidth=0.6, alpha=0.5, linestyle=":")
    ax.set_ylabel("Freq. (Hz)")
    ax.text(0.011, 0.86, s.name, transform=ax.transAxes, fontsize=10,
            fontweight="bold", color="w")

axes[-1].set_xlabel("Time (s)")
axes[0].set_title(f"Fig. 4a  Spectrograms, {NPERSEG}-sample Hann window, hop {HOP}"
                  f"  ({1000*NPERSEG/FS:.1f} ms, {FS/HOP:.0f} Hz frame rate)\n"
                  "dotted lines mark the 2000-4000 Hz band; shared colour scale, dB",
                  fontsize=11, pad=12)
fig4a.colorbar(im, ax=axes, label="Power (dB)", pad=0.015, aspect=40)
plt.show()

### 读图 4a —— 节奏第一次现身

**Healthy（第一行）**：整体暗，没有竖向结构、没有闪烁。4200–4500 Hz 有一条很淡的横带（就是步骤 03 里那几根窄线），但它**从头到尾亮度不变**。**这就是"随机噪声"的样子** —— 和步骤 02 测出的峭度 2.98 说的是同一件事。

**Outer race（第二行）**：**一条条明亮的竖条纹**，在两条白色虚线之间（2000–4000 Hz）最亮，间隔均匀。

**每一条竖纹就是一次撞击。** 你正在直接看着滚珠碾过那个坑。

这张图同时证实了步骤 03 的推理链：

- 条纹是**竖的**，说明一次撞击同时激起了一大片频率 → **冲击是宽带的**
- 条纹**周期性出现** → 那就是 105.9 Hz 的节奏，它一直都在，只是 FFT 把它平均掉了
- 亮度集中在 2–4 kHz → 那是轴承座的**固有共振频率**，撞击把它"敲响"

**Inner race（第三行）**：也有条纹，但**明暗不均** —— 有的亮有的淡。因为内圈跟着轴转，坑一会儿转到承重区、一会儿转开，撞击力道随之起伏。这正是图 1 里"内圈节奏糊"的原因，在这里看得一清二楚。

**Ball（第四行）**：这一行值得仔细看，它和你可能预期的不一样。

3000–3800 Hz 有一条**很亮的横带**，比健康那张亮得多 —— 所以"这个轴承不正常"是明确的。

**但那条带几乎不闪。** 它从左到右连续发亮，看不出竖向的分段。

**这个区别正是滚珠故障诊断不出来的原因，而且它把"检测"和"诊断"的分界线画得再清楚不过：**

| | 能量升高了吗 | 能量周期性起伏吗 |
|---|---|---|
| 检测"有故障" | 靠这个 | |
| 诊断"哪里坏" | | 靠这个 |

滚珠故障**有第一样，没有第二样**。所以步骤 02 的 RMS 抓得住它（0.139 vs 健康 0.062），而任何依赖节奏的方法都抓不住。

原因在物理：坑长在滚珠**自己身上**，滚珠一边公转一边自转，坑的朝向一直在变；撞击时打内圈、时打外圈，力的方向和大小都在变。**结果是能量持续泄出，但泄得没有节拍。**

### 自己验证一下

0.1 秒里，外圈应该出现 **105.9 × 0.1 ≈ 10.6 条**竖纹。数一数第二行。

**这是你第二次用眼睛测出 BPFO** —— 第一次是图 1 上数簇。两次独立的观察给出同一个数，而且都对上了从轴承几何算出来的理论值。

---
## 5. 把"闪烁"变成一条曲线

眼睛看到条纹了，下一步是**量出来**。

做法直接得出奇：**把每一帧里 2000–4000 Hz 的能量加起来，得到一个数。一帧一个数，排成一条曲线。**

这条曲线就是"高频能量随时间的起伏" —— 也就是那个**闪烁**本身。

In [ ]:
BAND = (2000, 4000)

def band_energy(name, band=BAND):
    f, t, P = grams[name]
    m = (f >= band[0]) & (f <= band[1])
    return t, P[m].sum(axis=0)

fig4b, axes = plt.subplots(4, 1, figsize=(11, 7.5), sharex=True)
COLORS = {"Healthy": "#256049", "Outer race": "#8E3320",
          "Inner race": "#1B4F8F", "Ball": "#9A5F0A"}

for ax, s in zip(axes, signals):
    t, e = band_energy(s.name)
    m = (t >= T0) & (t <= T1)
    ax.plot(t[m], e[m], linewidth=1.0, color=COLORS[s.name])
    ax.set_ylabel("Energy")
    ax.grid(alpha=0.25)
    ax.text(0.011, 0.80, s.name, transform=ax.transAxes,
            fontsize=10, fontweight="bold")

axes[-1].set_xlabel("Time (s)")
axes[0].set_title(f"Fig. 4b  Energy in {BAND[0]}-{BAND[1]} Hz over time "
                  f"(each panel on its own scale)", fontsize=11, pad=10)
fig4b.tight_layout()
plt.show()

外圈那条现在是一串**清清楚楚、间隔均匀的脉冲**。健康那条是一团没有结构的起伏。

> 注意这四个子图**各用各的纵轴**（没有 `sharey`）。这里是故意的 —— 我们关心的是"有没有周期结构"这个形状问题，不是幅度大小。幅度大小图 4a 的统一色标已经回答过了。**什么时候该统一坐标、什么时候不该，取决于你想让读者比较什么。**

## 你刚刚做出了一个粗糙的包络

这条曲线有个名字：**包络**（envelope）—— 它描的是原信号"外轮廓"的起伏，把里面 3000 Hz 的高频振荡抹掉了，只留下"一阵一阵"的节奏。

**既然它是一条随时间变化的曲线，就可以对它做 FFT。**

如果节奏真的是 105.9 Hz，那么这条曲线的频谱里就应该有一根 105.9 Hz 的峰。

In [ ]:
def band_energy_spectrum(name, band=BAND):
    t, e = band_energy(name, band)
    e = e - e.mean()                      # 去直流：能量恒正，不去均值 0 Hz 会有巨峰
    n = len(e)
    dt = t[1] - t[0]
    f = np.fft.rfftfreq(n, dt)
    A = np.abs(np.fft.rfft(e * np.hanning(n))) / n * 4
    return f, A


def peak_ratio(f, A, f0, tol=2.5):
    # 峰高 / 周围背景的中位数。衡量"这根峰突出不突出"
    near = np.abs(f - f0) <= tol
    bg = (np.abs(f - f0) > 10) & (np.abs(f - f0) < 60)
    return A[near].max() / np.median(A[bg]), f[near][np.argmax(A[near])]


fig4c, axes = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True)
LINES = [("BPFO", "#8E3320"), ("BPFI", "#1B4F8F"), ("BSF", "#9A5F0A")]

for ax, s in zip(axes, signals):
    f, A = band_energy_spectrum(s.name)
    m = f <= 500
    ax.plot(f[m], A[m] / A[m].max(), linewidth=0.8, color=COLORS[s.name])
    ff = s.fault_freqs()
    for k, c in LINES:
        ax.axvline(ff[k], color=c, linestyle="--", linewidth=1.1, alpha=0.8)
    ax.set_ylabel("Norm. amp.")
    ax.set_ylim(0, 1.08)
    ax.grid(alpha=0.25)
    ax.text(0.011, 0.80, s.name, transform=ax.transAxes,
            fontsize=10, fontweight="bold")

ff = signals[0].fault_freqs()
for k, c in LINES:
    axes[0].text(ff[k], 1.14, f"{k}\n{ff[k]:.0f}", color=c, fontsize=8,
                 ha="center", va="bottom", linespacing=1.15)

axes[-1].set_xlabel("Frequency (Hz)")
axes[-1].set_xlim(0, 500)
axes[0].set_title("Fig. 4c  Spectrum of the band energy  -  a crude envelope spectrum\n"
                  "each panel normalised to its own maximum",
                  fontsize=11, pad=34)
fig4c.tight_layout()
plt.show()

---
## 6. 数字：这根峰到底有多突出

肉眼说"有峰"不够。用**峰背比**量化：那根峰的高度，除以它周围背景的中位数。

- 峰背比 < 5 → 基本等于没有
- 峰背比 > 20 → 明确的峰

**最重要的是把健康记录也算一遍** —— 它给出这个指标的"误报底线"。

In [ ]:
print("每条记录在【它自己的】理论故障频率上的峰背比\n")
print(f"{'记录':<13}{'查哪个':<8}{'理论 Hz':>10}{'峰背比':>10}{'实测峰 Hz':>12}")
print("-" * 55)
for s in signals:
    key = TARGET[s.name]
    f0 = s.fault_freqs()[key]
    f, A = band_energy_spectrum(s.name)
    r, fp = peak_ratio(f, A, f0)
    print(f"{s.name:<13}{key:<8}{f0:>10.1f}{r:>10.1f}{fp:>12.1f}")

print("\n\n交叉检验：每条记录在三个频率上各是多少")
print("（对角线应该最大 —— 那才叫'诊断'）\n")
print(f"{'':<13}{'BPFO':>10}{'BPFI':>10}{'BSF':>10}")
print("-" * 45)
for s in signals:
    f, A = band_energy_spectrum(s.name)
    ff = s.fault_freqs()
    vals = [peak_ratio(f, A, ff[k])[0] for k in ("BPFO", "BPFI", "BSF")]
    print(f"{s.name:<13}" + "".join(f"{v:>10.1f}" for v in vals))

print("\n\n外圈的谐波列（真信号应该有 2 倍、3 倍频）\n")
f, A = band_energy_spectrum("Outer race")
bpfo = outer.fault_freqs()["BPFO"]
for k in range(1, 5):
    r, fp = peak_ratio(f, A, k * bpfo)
    print(f"  {k} x BPFO = {k*bpfo:6.1f} Hz    峰背比 {r:7.1f}    实测 {fp:6.1f} Hz")

---
## 7. 结果

### 外圈：成了，而且成得非常干净

| | 峰背比 |
|---|---|
| 1 × BPFO = 105.9 Hz | **601** |
| 2 × BPFO = 211.9 Hz | **504** |
| 3 × BPFO = 317.8 Hz | **371** |
| 4 × BPFO = 423.7 Hz | **325** |

**一整列谐波，全部极其突出。** 这比单独一根峰有说服力得多 —— 随机噪声不会凑巧在 1、2、3、4 倍频上同时冒出峰。

而且实测峰在 106.3 Hz，理论 105.9 Hz，**差 0.4%**。

对比一下步骤 03：同一份数据、同一个频率，直接 FFT 的峰背比连 1 都不到，现在是 601。**东西一直都在，只是之前用错了工具。**

### 内圈：也成了

BPFI 处峰背比 **222**，实测 159.6 Hz vs 理论 159.9 Hz。

**特别注意交叉检验那张表**：内圈记录在 BPFO 处只有 3.3，在 BSF 处 3.6，只有 BPFI 那一格是 222。

**这才叫诊断** —— 不是"某处有峰"，而是"**恰好在理论预测的那一处、而且只在那一处**有峰"。对角线亮、其余暗，这张表本身就是证据。

### 健康：确认了误报底线

健康记录在三个频率上是 3.6 / 7.2 / 2.3。**这告诉你：峰背比 7 以内的东西什么都不能说明。**

没有这一行，你就不知道 222 算不算多。**对照组第二次救了这个项目。**

### 滚珠：失败

BSF 处峰背比 **2.3** —— 和健康记录的 2.3 一模一样。检查 2 倍频（278.4 Hz）也只有 7.4。

**这条诊断不出来。** 老实说，后面用更好的方法（包络谱）和更合适的频带，它最多也就爬到 11 左右，而健康记录能刷到 7.9。**它很可能到项目结束都立不住。**

这不是你做错了 —— **图 4a 第四行已经把原因画出来了**：能量升高了（亮带很明显），但那条带不闪。有能量，没节拍。

而所有频率类方法量的都是节拍。

**这要如实写进报告的 Limitations。** 写"三种故障中两种诊断成功，滚珠故障在本方法下不可靠，原因是……"，比写"三种全部成功"可信得多 —— 后者反而会让懂行的人怀疑你是不是调参调出来的。

---
## 8. 这一节走到哪了

| | 步骤 03（直接 FFT） | 步骤 04（时频 + 带内能量） |
|---|---|---|
| 外圈 BPFO | 埋掉了 | **601**，含四次谐波 |
| 内圈 BPFI | 被机器谱线冒充 | **222**，交叉检验干净 |
| 滚珠 BSF | 无 | 2.3，仍然无 |
| 健康（误报底线） | — | 3.6 / 7.2 / 2.3 |

**思路上的转折就是这一句：**

> 不要在原信号的频谱里找 105.9 Hz。
> 要先取出高频共振那一段，看它的**强度如何随时间起伏**，再对这条起伏曲线做 FFT。

这正是**包络分析**的全部思想，你已经用 STFT 手工做了一遍粗糙版。

### 那还需要步骤 05 做什么

现在这个做法有三个明显的粗糙之处：

1. **时频分辨率被窗长绑死。** 窗 64 点 → 频率只能分到 187.5 Hz，想选 2000–2500 Hz 这样的窄带根本做不到
2. **包络采样率被 hop 绑死。** 想要更细的包络，就得算更多帧，计算量上去
3. **"2000–4000 Hz" 是我随手定的。** 凭什么是这一段？

步骤 05 用**带通滤波 + 希尔伯特变换**把前两条一次解决 —— 它直接在原始 12000 Hz 的时间分辨率上求包络，不需要分帧，频带想多窄就多窄。

第三条留给**步骤 06**，那是整个项目里唯一需要你自己做判断的地方，也是最值钱的一步。我上面偷看过：**内圈的最佳频带和外圈不一样**（1000–2000 vs 2500–3500）。这个发现会成为第 06 步的核心。

---
## 9. 保存

In [ ]:
FIGURES.mkdir(exist_ok=True)
for fig, name in [(fig4a, "fig04a_spectrogram.png"),
                  (fig4b, "fig04b_band_energy.png"),
                  (fig4c, "fig04c_band_energy_spectrum.png")]:
    p = FIGURES / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    print("已保存:", p)

p = FIGURES / "table03_stft_peak_ratios.txt"
with open(p, "w", encoding="utf-8") as fh:
    fh.write("Peak-to-background ratio in the spectrum of the "
             f"{BAND[0]}-{BAND[1]} Hz band energy\n")
    fh.write(f"STFT: {NPERSEG}-sample Hann window, hop {HOP} "
             f"({FS/HOP:.0f} Hz frame rate)\n\n")
    fh.write(f"{'':<14}{'BPFO':>10}{'BPFI':>10}{'BSF':>10}\n")
    fh.write("-" * 46 + "\n")
    for s in signals:
        f, A = band_energy_spectrum(s.name)
        ff = s.fault_freqs()
        vals = [peak_ratio(f, A, ff[k])[0] for k in ("BPFO", "BPFI", "BSF")]
        fh.write(f"{s.name:<14}" + "".join(f"{v:>10.1f}" for v in vals) + "\n")
    fh.write("\nThe healthy record sets the false-alarm floor for this metric.\n")
print("已保存:", p)

---
## 10. 自己动手

这次的练习**全是独立的格子，直接运行就行**，不用回去改上面的代码。想试别的值，改格子最上面那一行。

In [ ]:
# 练习 1：为什么不能用窗256/重叠50%
# ------------------------------------------------
# 改这一行试别的组合:
NPERSEG_TEST, HOP_TEST = 256, 128

f, t, Z = stft(outer.x[:N], FS, nperseg=NPERSEG_TEST,
               noverlap=NPERSEG_TEST - HOP_TEST, window="hann",
               boundary=None, padded=False)
P = np.abs(Z) ** 2
e = P[(f >= 2000) & (f <= 4000)].sum(axis=0)
e = e - e.mean()
dt = t[1] - t[0]
fr = np.fft.rfftfreq(len(e), dt)

print(f"窗长 {NPERSEG_TEST}, hop {HOP_TEST}")
print(f"  包络采样率      {1/dt:8.1f} Hz")
print(f"  频率轴最高只到  {fr.max():8.1f} Hz")
print(f"  要找的 BPFO     {BPFO:8.1f} Hz")
print()
if fr.max() < BPFO:
    print("  -> BPFO 超出了频率轴，它在这套参数下【根本不存在】")
    print("     这不是'看不清'，是信息已经被彻底丢掉了")
else:
    A = np.abs(np.fft.rfft(e * np.hanning(len(e)))) / len(e) * 4
    r, fp = peak_ratio(fr, A, BPFO)
    print(f"  -> BPFO 峰背比 {r:.1f}")

In [ ]:
# 练习 2：换一个频带，看外圈的峰背比怎么变
# ------------------------------------------------
# 这就是步骤 06 要系统做的事，先感受一下
BANDS_TO_TRY = [(500, 1500), (1000, 2000), (2000, 3000), (3000, 4000), (4000, 5000)]

print(f"{'频带':<16}{'Outer':>10}{'Inner':>10}{'Ball':>10}{'Healthy':>10}")
print("-" * 58)
for bd in BANDS_TO_TRY:
    row = []
    for s in signals[1:] + [signals[0]]:
        f, A = band_energy_spectrum(s.name, band=bd)
        row.append(peak_ratio(f, A, s.fault_freqs()[TARGET[s.name]])[0])
    print(f"{str(bd):<16}" + "".join(f"{v:>10.1f}" for v in row))

print("\n三个问题想一想：")
print("  1. 外圈最好的频带是哪个？和我默认用的 2000-4000 一样吗？")
print("  2. 内圈最好的频带和外圈一样吗？")
print("  3. 健康记录在某些频带也能刷出不低的值 —— 这说明峰背比这个指标有什么问题？")

In [ ]:
# 练习 3：改显示的时间窗，数竖纹
# ------------------------------------------------
# 改这一行: 0.05 秒应该有约 5.3 条，0.2 秒约 21 条
T_START, T_LEN = 0.10, 0.05

f, t, P = grams["Outer race"]
m = (t >= T_START) & (t <= T_START + T_LEN)

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.pcolormesh(t[m], f, 10 * np.log10(P[:, m] + 1e-20),
              vmin=vmin, vmax=vmax, cmap="magma", shading="auto")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Freq. (Hz)")
ax.set_title(f"Outer race, {T_LEN*1000:.0f} ms  "
             f"-  expect about {BPFO*T_LEN:.1f} stripes", fontsize=10)
plt.show()

print(f"理论条纹数 = BPFO x 时长 = {BPFO:.1f} x {T_LEN} = {BPFO*T_LEN:.1f}")

---
## 小结

| 做了 | 结论 |
|---|---|
| 由目标频率反推 STFT 参数 | 常见的"窗256/重叠50%"会把 105.9 Hz 直接抹掉 |
| 时频图（图 4a） | **看见了周期性竖条纹** —— 撞击本身 |
| 带内能量曲线（图 4b） | 把闪烁变成一条可分析的曲线 = 粗糙的包络 |
| 对包络做 FFT（图 4c） | 外圈 **601**（含 4 次谐波）、内圈 **222** |
| 交叉检验 | 对角线亮、其余暗 —— 这才是诊断 |
| 健康对照 | 误报底线 3.6 / 7.2 / 2.3 |
| 滚珠 | **2.3，失败**，物理上就难，如实写进 Limitations |

**核心转折**：不在原信号里找 105.9 Hz，而是**先取高频共振带，看它的强度如何起伏，再对起伏做 FFT**。

**下一步（步骤 05）**：用带通滤波 + 希尔伯特变换做真正的包络谱。三行代码，但它把这一节的所有粗糙之处一次解决 —— 并且把"频带该选哪一段"这个问题赤裸裸地摆到你面前。